# Spoor 1 – Tevredenheid met het leven, toekomstperspectief en grip

**Vraag:** wat is de relatie tussen tevredenheid met het leven, perspectief op de toekomst en gevoel van invloed/grip?

| Subvraag | Tabbladen in Excel |
|---|---|
| 1. Hangt (on)tevredenheid samen met de blik op de toekomst? (correlatie/regressie) | 1a–1f |
| 2. Welke groepen (bijv. ontevreden & pessimistisch), waarover pessimistisch, wie zitten erin? | 2a–2g |
| 3. Grotere kans op pessimisme als basisdingen niet op orde zijn? | 3a–3b |
| 4. Wat zijn basisdingen? | 4 |
| 5. Zekerheid nu vs. zekerheid in de toekomst | 5a–5c |
| 6. Invloed / grip | 6a–6d |
| Extra: kansenkaart, redenen pessimisme/optimisme | 7a–7c |

Output: `spoor1_factsheet_data.xlsx` (voor de vormgever), `spoor1_open_antwoorden_INTERN.xlsx` (citaten, niet delen), een paar PNG's om snel te checken.
Alles gewogen met `weegfactor`; n = ongewogen.

In [ ]:
import re
import textwrap
import numpy as np
import pandas as pd
import pyreadstat
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

path = r"O:\Projecten lopend\BW & kansengelijkheid\Data\Werkbestand BW en kansengelijkheid.sav"
EXCEL_OUT = "spoor1_factsheet_data.xlsx"
OPEN_OUT  = "spoor1_open_antwoorden_INTERN.xlsx"

weight_col        = "weegfactor"
TEVREDEN_GRENS    = 7   # cijfer >= 7 -> tevreden
ONVOLDOENDE_GRENS = 6   # cijfer <  6 -> domein niet op orde
LEEFTIJD_BINS     = [17, 34, 49, 64, 200]
LEEFTIJD_LABELS   = ["18-34 jaar", "35-49 jaar", "50-64 jaar", "65 jaar en ouder"]

GROEPEN = ["Tevreden & hoopvol/stabiel", "Tevreden & pessimistisch",
           "Ontevreden & hoopvol/stabiel", "Ontevreden & pessimistisch"]
TOTAAL = "Totaal"
colors_groep = dict(zip(GROEPEN, ["#2ecc71", "#f39c12", "#3498db", "#e74c3c"]))

DOMEIN_KORT = [
    ("leven in het algemeen", "Leven in het algemeen"),
    ("financi", "Financiële situatie"),
    ("lichamelijke gezondheid", "Lichamelijke gezondheid"),
    ("psychische gezondheid", "Psychische gezondheid"),
    ("werk en/of andere", "Werk/dagelijkse bezigheden"),
    ("vrije tijd", "Vrije tijd"),
    ("opleidingskansen", "Opleidingskansen"),
    ("woning of woonruimte", "Woning"),
    ("directe woonomgeving", "Buurt/wijk"),
    ("sociale contacten", "Sociale contacten"),
    ("met elkaar omgaan", "Omgang in de buurt"),
    ("veiligheid", "Veiligheid in de buurt"),
    ("groen en de natuur", "Groen en natuur"),
    ("milieukwaliteit", "Milieukwaliteit"),
    ("bereikbaarheid", "Bereikbaarheid voorzieningen"),
    ("culturele aanbod", "Cultureel aanbod"),
]
LEVEN = "Leven in het algemeen"

TOEKOMST_MAP    = {"veel slechter": 1, "een beetje slechter": 2, "hetzelfde": 3,
                   "een beetje beter": 4, "veel beter": 5}
TOEKOMST_LABELS = ["Veel slechter", "Een beetje slechter", "Hetzelfde", "Een beetje beter", "Veel beter"]
INVLOED_MAP     = {"geen invloed": 0, "een beetje invloed": 1, "veel invloed": 2}
INVLOED_LABELS  = {0: "Geen invloed", 1: "Een beetje invloed", 2: "Veel invloed"}

tabellen = {}   # alles wat naar Excel gaat

## Hulpfuncties

In [ ]:
# ── Inlezen en hercoderen ───────────────────────────────────────────────────
VALUE_ORDER = {}   # volgorde SPSS-waardelabels, voor nette tabellen

def load_and_label(path):
    df, meta = pyreadstat.read_sav(path)
    label_map = {c: l for c, l in zip(meta.column_names, meta.column_labels) if l}
    df.rename(columns=label_map, inplace=True)
    for col in meta.column_names:
        if col in meta.variable_value_labels:
            new = label_map.get(col, col)
            value_map = meta.variable_value_labels[col]
            df[new] = df[new].map(value_map).fillna(df[new])
            VALUE_ORDER[new] = list(dict.fromkeys(value_map.values()))
    cols = pd.Series(df.columns)
    for dup in cols[cols.duplicated()].unique():
        for i, idx in enumerate(cols[cols == dup].index):
            if i > 0:
                cols[idx] = f"{dup}__{i+1}"
    df.columns = cols
    return df

def clean_label(c):
    return re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", str(c))).strip()

def subject(c):
    return clean_label(c).split(" - ")[-1].strip()

def find_cols(df, *needles, exclude=()):
    return [c for c in df.columns
            if all(n.lower() in clean_label(c).lower() for n in needles)
            and not any(e.lower() in clean_label(c).lower() for e in exclude)]

def first_col(df, *kandidaten):
    for k in kandidaten:
        if k in df.columns:
            return k
    for k in kandidaten:
        if find_cols(df, k):
            return find_cols(df, k)[0]
    return None

def kort(s):
    return next((naam for key, naam in DOMEIN_KORT if key in s.lower()), s[:45])

def norm(s):
    return s.astype("string").str.strip().str.lower()

def to_score(s):
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(float)
    lead = pd.to_numeric(s.astype("string").str.extract(r"^\s*(\d{1,2})")[0], errors="coerce")
    return pd.to_numeric(s, errors="coerce").fillna(lead).astype(float)

def to_likert(s):
    # (helemaal) oneens = 1 ... (helemaal) eens = 5, weet niet = NaN
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(float)
    def m(x):
        if pd.isna(x):
            return np.nan
        x = str(x).lower().replace("mee ", "")
        if "weet" in x or "wil ik niet" in x:   return np.nan
        if "neutraal" in x or ("niet eens" in x and "niet oneens" in x): return 3
        if "helemaal oneens" in x or "zeer oneens" in x: return 1
        if "helemaal eens" in x or "zeer eens" in x:     return 5
        if "oneens" in x: return 2
        if "eens" in x:   return 4
        return np.nan
    return s.map(m).astype(float)

def to_janee(s):
    # ja = 1, nee/weet ik niet = 0 (geen zekere steun)
    if pd.api.types.is_numeric_dtype(s):
        return (s > 0).astype(float).where(s.notna())
    x = norm(s)
    out = pd.Series(np.nan, index=s.index)
    out[x.str.startswith("ja", na=False)] = 1
    out[x.str.startswith("nee", na=False) | x.str.contains("weet", na=False)] = 0
    return out

def binary_block(df, cols):
    # aanvinkvragen -> 0/1; wie niets in het blok invulde -> NaN
    out = pd.DataFrame(index=df.index)
    for c in cols:
        s = df[c]
        if pd.api.types.is_numeric_dtype(s):
            out[c] = (s > 0).astype(float).where(s.notna())
        else:
            x = norm(s)
            v = pd.Series(np.nan, index=s.index)
            v[x.isin(["0", "0.0", "nee", "niet geselecteerd", "not selected", "false"])] = 0
            v[x.notna() & v.isna()] = 1
            out[c] = v
    beantwoord = out.notna().any(axis=1)
    return out.apply(lambda col: col.fillna(0).where(beantwoord))

def ind(cond, s):
    # 0/1-indicator, NaN waar s mist
    return cond.astype(float).where(s.notna())

def ordered_cats(df, col):
    present = list(df[col].dropna().unique())
    order = [v for v in VALUE_ORDER.get(col, []) if v in present]
    return order + sorted([v for v in present if v not in order], key=str)

In [ ]:
# ── Gewogen statistiek, modellen en opslaan ─────────────────────────────────
def wmean(x, w):
    m = x.notna() & w.notna()
    return float(np.average(x[m].astype(float), weights=w[m].astype(float))) if m.any() else np.nan

def wpct(x, w):
    v = wmean(x, w)
    return np.nan if pd.isna(v) else 100 * v

def wcorr(x, y, w):
    m = x.notna() & y.notna() & w.notna()
    if m.sum() < 10:
        return np.nan, int(m.sum())
    x, y, w = x[m].astype(float), y[m].astype(float), w[m].astype(float)
    mx, my = np.average(x, weights=w), np.average(y, weights=w)
    cov = np.average((x - mx) * (y - my), weights=w)
    return float(cov / np.sqrt(np.average((x - mx)**2, weights=w) * np.average((y - my)**2, weights=w))), int(m.sum())

def z(s):
    s = s.astype(float)
    return (s - s.mean()) / s.std()

def fit_wls(y, X, w):
    d = pd.concat([y.rename("_y"), X, w.rename("_w")], axis=1).dropna()
    return sm.WLS(d["_y"], sm.add_constant(d[X.columns], has_constant="add"), weights=d["_w"]).fit(), len(d)

def fit_logit(y, X, w):
    d = pd.concat([y.rename("_y"), X, w.rename("_w")], axis=1).dropna()
    model = sm.GLM(d["_y"], sm.add_constant(d[X.columns], has_constant="add"),
                   family=sm.families.Binomial(), freq_weights=d["_w"] / d["_w"].mean())
    return model.fit(), len(d)

def per_groep(fn):
    # fn(mask) per groep + totaal (totaal = alle ingedeelde respondenten)
    out = {g: fn(df["_groep"] == g) for g in GROEPEN}
    out[TOTAAL] = fn(df["_groep"].notna())
    return out

def gw(x):
    # gewogen % van 0/1-reeks binnen elke groep
    return per_groep(lambda m: wpct(x[m], w[m]))

def n_rij(**extra):
    return {**extra, **{g: int((df["_groep"] == g).sum()) for g in GROEPEN},
            TOTAAL: int(df["_groep"].notna().sum())}

def bewaar(naam, titel, vraag, tb, toelichting="", fmt=None, visual=""):
    tabellen[naam[:31]] = dict(titel=titel, vraag=vraag, df=tb.reset_index(drop=True),
                               toelichting=toelichting, fmt=fmt or {}, visual=visual)
    display(tb.round(2))

## Data inlezen

In [ ]:
df = load_and_label(path)
print(f"Ingelezen: {df.shape}")
df = df[df["Reactiestatus"] != "Niet geantwoord"].copy()
df = df.loc[:, ~df.columns.duplicated()]
print(f"Na filter Reactiestatus: {df.shape}")

## Variabelen opbouwen

In [ ]:
# ── 1. Tevredenheid nu en verwachting komende 5 jaar per domein ─────────────
df[weight_col] = pd.to_numeric(df[weight_col], errors="coerce")
w = df[weight_col]

tev_cols = find_cols(df, "hoe tevreden u bent met")
toe_cols = [c for c in find_cols(df, "komende 5 jaar") if clean_label(c).lower().startswith("hoe denkt u")]

tev = pd.DataFrame({kort(subject(c)): to_score(df[c]) for c in tev_cols})
toe = pd.DataFrame({kort(subject(c)): norm(df[c]).map(TOEKOMST_MAP).astype(float) for c in toe_cols})
domeinen = [k for _, k in DOMEIN_KORT if k in tev.columns]
domeinen_excl_leven = [d for d in domeinen if d != LEVEN]

print("=== Gevonden ===")
print(f"Tevredenheid: {len(tev_cols)} | Toekomst: {len(toe_cols)} | Weegfactor geldig: {w.notna().sum()}, gem. {w.mean():.3f}")
print("Domeinen:", domeinen)
print("\n=== Toekomst leven: ruw -> numeriek (check!) ===")
c_leven = [c for c in toe_cols if kort(subject(c)) == LEVEN][0]
print(pd.crosstab(df[c_leven].fillna("(leeg)"), toe[LEVEN].fillna(-1)))

In [ ]:
# ── 2. Ervaren invloed (grip) en stellingen ─────────────────────────────────
inv_raw = {kort(subject(c)): c for c in find_cols(df, "hoeveel invloed")}
inv = pd.DataFrame({k: norm(df[c]).map(INVLOED_MAP).astype(float) for k, c in inv_raw.items()})

stel_cols = find_cols(df, "in hoeverre u het eens bent")
stel = pd.DataFrame({subject(c): to_likert(df[c]) for c in stel_cols})

stel_kort = []
for naam, key in [("Zelf veel invloed op hoe leven verloopt", "invloed heb op hoe mijn leven"),
                  ("Neemt zelf belangrijke beslissingen voor toekomst", "zelf beslissingen"),
                  ("Zelf minstens evenveel kansen gekregen", "evenveel kansen gekregen"),
                  ("In Nederland heeft iedereen evenveel kansen", "iedereen evenveel kansen")]:
    c = next((c for c in stel.columns if key in c.lower()), None)
    if c:
        stel_kort.append((naam, c))
grip_stelling = stel_kort[0][1] if stel_kort and stel_kort[0][0].startswith("Zelf veel") else None

print("=== Invloed ===")
for k, c in inv_raw.items():
    print(f"{k}: {df[c].value_counts(dropna=False).to_dict()}")
print("\n=== Stellingen: ruw -> 1-5 (check hercodering!) ===")
for c in stel_cols:
    print(subject(c)[:70])
    print("  ", df[c].value_counts(dropna=False).to_dict())
    print("  ->", to_likert(df[c]).value_counts(dropna=False).sort_index().to_dict())

In [ ]:
# ── 3. Kernvariabelen en de vier groepen ────────────────────────────────────
df["_tev_leven"]         = tev[LEVEN]
df["_toe_leven"]         = toe[LEVEN]
df["_tevreden"]          = ind(df["_tev_leven"] >= TEVREDEN_GRENS, df["_tev_leven"])
df["_onvoldoende_leven"] = ind(df["_tev_leven"] < ONVOLDOENDE_GRENS, df["_tev_leven"])
df["_pessimistisch"]     = ind(df["_toe_leven"] < 3, df["_toe_leven"])
df["_optimistisch"]      = ind(df["_toe_leven"] > 3, df["_toe_leven"])
df["_inv_leven"]         = inv[LEVEN]
df["_veel_invloed"]      = ind(df["_inv_leven"] == 2, df["_inv_leven"])
df["_geen_invloed"]      = ind(df["_inv_leven"] == 0, df["_inv_leven"])

t, p = df["_tevreden"], df["_pessimistisch"]
conditions = [(t == 1) & (p == 0), (t == 1) & (p == 1), (t == 0) & (p == 0), (t == 0) & (p == 1)]
df["_groep"] = pd.Series(np.select(conditions, GROEPEN, default=None), index=df.index)

print("=== Groepen (ongewogen n) ===")
print(df["_groep"].value_counts(dropna=False))
q_lev = first_col(df, "Q_levenalgemeen")
if q_lev:
    print("\n=== Check: eigen indeling vs bestaande Q_levenalgemeen ===")
    print(pd.crosstab(df["_groep"].fillna("(leeg)"), df[q_lev].fillna("(leeg)")))

In [ ]:
# ── 4. Domeinen niet op orde, onzekerheden, netwerk ─────────────────────────
onv = pd.DataFrame({d: ind(tev[d] < ONVOLDOENDE_GRENS, tev[d]) for d in domeinen_excl_leven})
df["_n_onvoldoende"] = onv.sum(axis=1, min_count=1)

excl = ("geen van bovenstaande", "wil ik niet zeggen")
fin_cols = find_cols(df, "situaties zijn op u van toepassing", exclude=excl)
ovr_cols = find_cols(df, "situaties is op u van toepassing", exclude=excl)
onz_fin = binary_block(df, fin_cols).set_axis([subject(c) for c in fin_cols], axis=1)
onz_ovr = binary_block(df, ovr_cols).set_axis([subject(c) for c in ovr_cols], axis=1)
df["_n_onz_fin"] = onz_fin.sum(axis=1, min_count=1)

net_cols  = find_cols(df, "zou kunnen helpen met") + find_cols(df, "terechtkunt")
hulp_cols = find_cols(df, "welke organisaties ik terecht kan")
net  = pd.DataFrame({subject(c): to_janee(df[c]) for c in net_cols}, index=df.index)
hulp = pd.DataFrame({subject(c): to_janee(df[c]) for c in hulp_cols}, index=df.index)

print("=== Gevonden ===")
print(f"Financiële onzekerheden: {len(fin_cols)} | overige: {len(ovr_cols)} | netwerk: {len(net_cols)} | hulp: {len(hulp_cols)}")
print("\n=== Voorbeelden ruwe waarden ===")
for c in fin_cols[:1] + net_cols[:1] + hulp_cols[:1]:
    print(subject(c)[:50], df[c].value_counts(dropna=False).to_dict())
print("\nAantal domeinen onvoldoende:", df["_n_onvoldoende"].value_counts().sort_index().to_dict())

In [ ]:
# ── 5. Achtergrond, verleden en controlevariabelen ──────────────────────────
df["_leeftijd"] = pd.to_numeric(df["Leeftijd"], errors="coerce")
df["Leeftijdsgroep"] = pd.cut(df["_leeftijd"], LEEFTIJD_BINS, labels=LEEFTIJD_LABELS).astype("object")
VALUE_ORDER["Leeftijdsgroep"] = LEEFTIJD_LABELS

gesl = first_col(df, "Geslacht")
if gesl:
    df["Geslacht_clean"] = (df[gesl].astype("string").str.strip()
                            .replace({"Een man": "Man", "Een vrouw": "Vrouw"}).astype("object"))
    VALUE_ORDER["Geslacht_clean"] = ["Man", "Vrouw", "Non-binair", "Wil ik niet zeggen"]

achtergrond = [(naam, col) for naam, col in [
    ("Leeftijdsgroep",             "Leeftijdsgroep"),
    ("Geslacht",                   "Geslacht_clean" if gesl else None),
    ("Opleidingsniveau",           first_col(df, "opleiding3", "Opleiding (3 niveaus)")),
    ("Inkomen huishouden",         first_col(df, "Inkomencat", "Inkomen (categorie)")),
    ("Koop/huur",                  first_col(df, "Woont u in een koopwoning of huurwoning")),
    ("Huishouden",                 first_col(df, "Hoe kunt u uw huishouden het beste omschrijven")),
    ("Belangrijkste inkomensbron", first_col(df, "Uit welke bron komt het grootste deel")),
    ("Provincie",                  first_col(df, "Provincie", "regioprovincie")),
] if col]

verleden = [(naam, col) for naam, col in [
    ("Ouders konden goed rondkomen (jeugd)",         first_col(df, "Konden uw ouders of verzorgers over het algemeen goed rondkomen")),
    ("Ouders hadden voldoende tijd/aandacht (jeugd)", first_col(df, "Hadden uw ouders of verzorgers voldoende tijd en aandacht")),
    ("Kon als volwassene (25+) goed rondkomen",      first_col(df, "Kon u als volwassene (vanaf 25 jaar) meestal goed rondkomen")),
    ("Moeite met rondkomen afgelopen 12 maanden",    first_col(df, "afgelopen 12 maanden moeite gehad om van het inkomen")),
] if col]

dag_cols = find_cols(df, "in welke situatie(s) zit u", exclude=("wil ik niet zeggen",))
dagbesteding = binary_block(df, dag_cols).set_axis([subject(c) for c in dag_cols], axis=1)

opl_col, ink_col = dict(achtergrond).get("Opleidingsniveau"), dict(achtergrond).get("Inkomen huishouden")

# controlevariabelen regressies: leeftijd, geslacht, opleiding, inkomen
C = pd.DataFrame({"Leeftijd": df["_leeftijd"]})
if gesl:
    C["Vrouw"] = df["Geslacht_clean"].map({"Vrouw": 1.0, "Man": 0.0})
for col, prefix in [(opl_col, "Opleiding"), (ink_col, "Inkomen")]:
    if col:
        s = df[col].astype("object")
        s = s.where(~s.astype(str).str.lower().str.contains("weet|wil ik niet|onbekend", na=False))
        ref = s.value_counts().idxmax()
        d = pd.get_dummies(s, prefix=prefix, dtype=float).drop(columns=f"{prefix}_{ref}")
        d[s.isna()] = np.nan
        C = pd.concat([C, d], axis=1)
Cs = C.copy()
Cs["Leeftijd"] = z(Cs["Leeftijd"])

print("=== Achtergrond ===")
for naam, col in achtergrond + verleden:
    print(f"{naam}: {df[col].value_counts(dropna=False).head(10).to_dict()}")
print("\nDagbesteding:", list(dagbesteding.columns))
print("Controlevariabelen:", list(C.columns))

## Subvraag 1 – Hangt (on)tevredenheid samen met de blik op de toekomst?

In [ ]:
# ── 1a. Kerncijfers ─────────────────────────────────────────────────────────
VR1 = "1. Hangt (on)tevredenheid met het leven samen met de blik op de toekomst (en met grip)?"
r_tt, _ = wcorr(df["_tev_leven"], df["_toe_leven"], w)
r_ti, _ = wcorr(df["_tev_leven"], df["_inv_leven"], w)

rows = [
    ("Aantal respondenten (ongewogen)", len(df), "aantal", ""),
    ("Gemiddelde tevredenheid met het leven", wmean(df["_tev_leven"], w), "cijfer 1-10", ""),
    (f"Tevreden met het leven (cijfer {TEVREDEN_GRENS}+)", wpct(df["_tevreden"], w), "%", ""),
    (f"Onvoldoende voor het leven (cijfer <{ONVOLDOENDE_GRENS})", wpct(df["_onvoldoende_leven"], w), "%", ""),
    ("Verwacht dat het leven over 5 jaar beter is", wpct(df["_optimistisch"], w), "%", ""),
    ("Verwacht dat het leven over 5 jaar hetzelfde is", wpct(ind(df["_toe_leven"] == 3, df["_toe_leven"]), w), "%", ""),
    ("Verwacht dat het leven over 5 jaar slechter is (pessimistisch)", wpct(df["_pessimistisch"], w), "%", ""),
    ("Ervaart veel invloed op eigen leven", wpct(df["_veel_invloed"], w), "%", ""),
    ("Ervaart geen invloed op eigen leven", wpct(df["_geen_invloed"], w), "%", ""),
    ("Samenhang tevredenheid nu en toekomstverwachting (r)", r_tt, "correlatie", "0,1 zwak - 0,3 matig - 0,5 sterk"),
    ("Samenhang tevredenheid nu en ervaren invloed (r)", r_ti, "correlatie", ""),
]
for g in GROEPEN:
    rows.append((f"Groep: {g}", wpct(ind(df["_groep"] == g, df["_groep"]), w), "%", f"n = {(df['_groep'] == g).sum()}"))

bewaar("1a_Kerncijfers", "Kerncijfers tevredenheid, toekomstperspectief en grip", VR1,
       pd.DataFrame(rows, columns=["Indicator", "Waarde", "Eenheid", "Opmerking"]),
       "Gewogen. Basis voor de 'grote getallen' op het feitenblad.", fmt={"Waarde": "0.0"}, visual="Kerncijfers / iconen")

In [ ]:
# ── 1b. Toekomstverwachting per tevredenheidsniveau ─────────────────────────
banden = ["1-5", "6", "7", "8", "9-10"]
band = pd.cut(df["_tev_leven"], [0, 5, 6, 7, 8, 10], labels=banden).astype("object")
toe_lab = df["_toe_leven"].map(dict(enumerate(TOEKOMST_LABELS, start=1)))

rows = []
for b in banden:
    m = (band == b) & toe_lab.notna()
    row = {"Tevredenheid met leven (cijfer)": b}
    for lab in TOEKOMST_LABELS:
        row[lab] = 100 * w[m & (toe_lab == lab)].sum() / w[m].sum()
    row["% pessimistisch (slechter)"] = row["Veel slechter"] + row["Een beetje slechter"]
    row["% optimistisch (beter)"] = row["Een beetje beter"] + row["Veel beter"]
    row["n"] = int(m.sum())
    rows.append(row)
tb_1b = pd.DataFrame(rows)

bewaar("1b_Toekomst_per_tevredenheid", "Toekomstverwachting naar tevredenheid met het leven nu", VR1, tb_1b,
       "Rij-percentages (per rij 100%). 'Weet ik niet' bij toekomst niet meegeteld.",
       visual="100%-gestapelde staaf per cijfergroep")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
kleuren = ["#c0392b", "#e67e22", "#bdc3c7", "#82c785", "#27ae60"]
left = np.zeros(len(tb_1b))
for lab, kleur in zip(TOEKOMST_LABELS, kleuren):
    ax.barh(tb_1b["Tevredenheid met leven (cijfer)"], tb_1b[lab], left=left, color=kleur, label=lab, edgecolor="white")
    left += tb_1b[lab].values
ax.set_xlim(0, 100)
ax.set_xlabel("% (gewogen)")
ax.set_ylabel("Tevredenheid met het leven (cijfer)")
ax.set_title("Verwachting voor het leven over 5 jaar, naar tevredenheid nu", fontsize=14, fontweight="bold")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.32), ncol=5, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("spoor1_toekomst_per_tevredenheid.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 1c. Tevredenheid naar toekomstverwachting en ervaren invloed ────────────
rows = []
for i, lab in enumerate(TOEKOMST_LABELS, start=1):
    m = df["_toe_leven"] == i
    rows.append({"Kenmerk": "Verwachting leven over 5 jaar", "Categorie": lab,
                 "Gem. tevredenheid leven (1-10)": wmean(df.loc[m, "_tev_leven"], w[m]),
                 f"% tevreden ({TEVREDEN_GRENS}+)": wpct(df.loc[m, "_tevreden"], w[m]),
                 "% pessimistisch over leven": np.nan,
                 "% van alle respondenten": wpct(ind(m, df["_toe_leven"]), w),
                 "n": int(m.sum())})
for k, lab in INVLOED_LABELS.items():
    m = df["_inv_leven"] == k
    rows.append({"Kenmerk": "Ervaren invloed op eigen leven", "Categorie": lab,
                 "Gem. tevredenheid leven (1-10)": wmean(df.loc[m, "_tev_leven"], w[m]),
                 f"% tevreden ({TEVREDEN_GRENS}+)": wpct(df.loc[m, "_tevreden"], w[m]),
                 "% pessimistisch over leven": wpct(df.loc[m, "_pessimistisch"], w[m]),
                 "% van alle respondenten": wpct(ind(m, df["_inv_leven"]), w),
                 "n": int(m.sum())})

bewaar("1c_Tevredenheid_per_toekomst", "Tevredenheid met het leven naar toekomstverwachting en ervaren invloed", VR1,
       pd.DataFrame(rows), "Gewogen gemiddelden/percentages.",
       fmt={"Gem. tevredenheid leven (1-10)": "0.0"}, visual="Staafdiagram gemiddeld cijfer")

In [ ]:
# ── 1d. Samenhang per domein ────────────────────────────────────────────────
rows = []
for d in domeinen:
    r1, n1 = wcorr(tev[d], toe[d], w) if d in toe.columns else (np.nan, 0)
    r2, _  = wcorr(tev[d], inv[d], w) if d in inv.columns else (np.nan, 0)
    r3, _  = wcorr(tev[d], df["_tev_leven"], w) if d != LEVEN else (np.nan, 0)
    r4, _  = wcorr(toe[d], df["_toe_leven"], w) if d != LEVEN and d in toe.columns else (np.nan, 0)
    rows.append({"Domein": d,
                 "r tevredenheid nu x verwachting toekomst (zelfde domein)": r1,
                 "r tevredenheid nu x ervaren invloed (zelfde domein)": r2,
                 "r domein-tevredenheid x tevredenheid leven": r3,
                 "r domein-verwachting x verwachting leven": r4,
                 "n": n1})
tb = pd.DataFrame(rows)

bewaar("1d_Samenhang_per_domein", "Samenhang (correlatie) tussen tevredenheid nu, toekomst en invloed per domein", VR1, tb,
       "Gewogen Pearson-correlaties. 0,1 zwak - 0,3 matig - 0,5 sterk. Leeg = vraag niet gesteld voor dit domein.",
       fmt={c: "0.00" for c in tb.columns if c.startswith("r ")}, visual="Gesorteerde staaf of heatmap")

In [ ]:
# ── 1e. Regressie: wat voorspelt tevredenheid met het leven? ────────────────
preds = {"Verwachting leven over 5 jaar (1-5)": df["_toe_leven"],
         "Ervaren invloed op leven (0-2)": df["_inv_leven"]}
if grip_stelling:
    preds["Stelling: 'ik heb veel invloed op hoe mijn leven verloopt' (1-5)"] = stel[grip_stelling]

idx = pd.concat([df["_tev_leven"]] + list(preds.values()) + [C, w], axis=1).dropna().index
y   = z(df.loc[idx, "_tev_leven"])
Xp  = pd.DataFrame({k: z(s.loc[idx]) for k, s in preds.items()})
Cz  = C.loc[idx].copy()
Cz["Leeftijd"] = z(Cz["Leeftijd"])

modellen = {"Model 1: alleen achtergrond": [],
            "Model 2: + toekomstverwachting": list(preds)[:1],
            "Model 3: + invloed/grip": list(preds)}
res = {naam: fit_wls(y, pd.concat([Cz, Xp[p]], axis=1), w.loc[idx]) for naam, p in modellen.items()}

rows = []
for k in list(preds) + ["Leeftijd"]:
    row = {"Voorspeller": k}
    for naam, (mdl, n) in res.items():
        row[f"{naam} - beta"] = mdl.params.get(k, np.nan)
        row[f"{naam} - p"] = mdl.pvalues.get(k, np.nan)
    rows.append(row)
rows.append({"Voorspeller": "R² (verklaarde variantie)", **{f"{naam} - beta": mdl.rsquared for naam, (mdl, n) in res.items()}})
rows.append({"Voorspeller": "n", **{f"{naam} - beta": n for naam, (mdl, n) in res.items()}})
tb = pd.DataFrame(rows)

bewaar("1e_Regressie_tevredenheid", "Regressie: wat voorspelt tevredenheid met het leven?", VR1, tb,
       "Gewogen lineaire regressie (WLS), gestandaardiseerde coëfficiënten (beta): hoe groter, hoe sterker. Alle modellen "
       "gecorrigeerd voor leeftijd, geslacht, opleiding en inkomen; zelfde respondenten in alle modellen. p < 0,05 = significant. "
       "Toename R² model 1 -> 2 = extra verklaring door toekomstverwachting.",
       fmt={c: ("0.000" if c.endswith("- p") else "0.00") for c in tb.columns if c != "Voorspeller"},
       visual="Kernboodschap; evt. staaf R² per model")

In [ ]:
# ── 1f. Per leeftijdsgroep (jongeren) ───────────────────────────────────────
rows = []
for lg in LEEFTIJD_LABELS + [TOTAAL]:
    m = (df["Leeftijdsgroep"] == lg) if lg != TOTAAL else df["Leeftijdsgroep"].notna()
    r, _ = wcorr(df.loc[m, "_tev_leven"], df.loc[m, "_toe_leven"], w[m])
    row = {"Leeftijdsgroep": lg,
           "Gem. tevredenheid leven (1-10)": wmean(df.loc[m, "_tev_leven"], w[m]),
           f"% tevreden ({TEVREDEN_GRENS}+)": wpct(df.loc[m, "_tevreden"], w[m]),
           "% pessimistisch over leven": wpct(df.loc[m, "_pessimistisch"], w[m]),
           "% optimistisch over leven": wpct(df.loc[m, "_optimistisch"], w[m]),
           "% veel invloed op leven": wpct(df.loc[m, "_veel_invloed"], w[m])}
    for g in GROEPEN:
        row[f"% {g}"] = wpct(ind(df.loc[m, "_groep"] == g, df.loc[m, "_groep"]), w[m])
    row["r tevredenheid x toekomst"] = r
    row["n"] = int(m.sum())
    rows.append(row)

bewaar("1f_Per_leeftijdsgroep", "Tevredenheid, toekomstperspectief en grip per leeftijdsgroep", VR1, pd.DataFrame(rows),
       "Eén meetmoment: een trend ('steeds ontevredener') is met deze data niet te toetsen.",
       fmt={"Gem. tevredenheid leven (1-10)": "0.0", "r tevredenheid x toekomst": "0.00"},
       visual="Gegroepeerde staaf per leeftijdsgroep")

## Subvraag 2 – Welke groepen, waarover pessimistisch, wie zitten erin?

In [ ]:
# ── 2a. De vier groepen ─────────────────────────────────────────────────────
VR2 = "2. Welke groepen zijn er (bijv. ontevreden & pessimistisch), wie zitten erin en waarover zijn ze pessimistisch?"
rows = [n_rij(Kenmerk="n (ongewogen)"),
        {"Kenmerk": "% van de bevolking", **per_groep(lambda m: 100 * w[m].sum() / w[df["_groep"].notna()].sum())},
        {"Kenmerk": "Gem. tevredenheid leven (1-10)", **per_groep(lambda m: wmean(df.loc[m, "_tev_leven"], w[m]))},
        {"Kenmerk": "Gem. toekomstverwachting leven (1=veel slechter, 5=veel beter)", **per_groep(lambda m: wmean(df.loc[m, "_toe_leven"], w[m]))},
        {"Kenmerk": "% veel invloed op eigen leven", **gw(df["_veel_invloed"])},
        {"Kenmerk": "% geen invloed op eigen leven", **gw(df["_geen_invloed"])},
        {"Kenmerk": f"Gem. aantal domeinen onvoldoende (<{ONVOLDOENDE_GRENS})", **per_groep(lambda m: wmean(df.loc[m, "_n_onvoldoende"], w[m]))},
        {"Kenmerk": "Gem. aantal financiële onzekerheden", **per_groep(lambda m: wmean(df.loc[m, "_n_onz_fin"], w[m]))},
        {"Kenmerk": "Gem. leeftijd", **per_groep(lambda m: wmean(df.loc[m, "_leeftijd"], w[m]))}]
if grip_stelling:
    rows.append({"Kenmerk": "% (helemaal) eens: 'ik heb veel invloed op hoe mijn leven verloopt'",
                 **gw(ind(stel[grip_stelling] >= 4, stel[grip_stelling]))})
tb_2a = pd.DataFrame(rows)

bewaar("2a_Groepen_overzicht", "De vier groepen: tevredenheid nu x verwachting voor de toekomst", VR2, tb_2a,
       f"Tevreden = cijfer {TEVREDEN_GRENS}+ voor 'leven in het algemeen'. Pessimistisch = verwacht dat het leven over 5 jaar "
       "(een beetje of veel) slechter is; hoopvol/stabiel = hetzelfde of beter. 'Weet ik niet' = niet ingedeeld.",
       visual="Kwadrant/2x2-figuur met groepsgroottes")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
pct = tb_2a.loc[1, GROEPEN].astype(float)
ax.barh(GROEPEN[::-1], pct[GROEPEN[::-1]], color=[colors_groep[g] for g in GROEPEN[::-1]])
for i, v in enumerate(pct[GROEPEN[::-1]]):
    ax.text(v + 0.5, i, f"{v:.0f}%", va="center", fontsize=10)
ax.set_xlabel("% van de bevolking (gewogen)")
ax.set_title("Tevredenheid nu x verwachting voor de toekomst", fontsize=14, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("spoor1_groepen.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 2b. Op welke vlakken pessimistisch? ─────────────────────────────────────
rows = [{"Domein": d, **gw(ind(toe[d] < 3, toe[d]))} for d in domeinen_excl_leven if d in toe.columns]
tb = pd.DataFrame(rows)
tb["Verschil ontevreden&pess. vs totaal (%-punt)"] = tb[GROEPEN[3]] - tb[TOTAAL]
tb = tb.sort_values(GROEPEN[3], ascending=False)

bewaar("2b_Pessimisme_per_domein", "Op welke vlakken zijn groepen pessimistisch? (% verwacht 'slechter' over 5 jaar)", VR2, tb,
       "Gewogen % dat verwacht dat het op dit domein (een beetje of veel) slechter gaat. Gesorteerd op 'Ontevreden & pessimistisch'.",
       visual="Dot-plot per domein")

In [ ]:
tb = tabellen["2b_Pessimisme_per_domein"]["df"].sort_values(GROEPEN[3])
ypos = np.arange(len(tb))

fig, ax = plt.subplots(figsize=(10, 7))
ax.hlines(ypos, tb[GROEPEN].min(axis=1), tb[GROEPEN].max(axis=1), color="#dddddd", linewidth=2, zorder=1)
for g in GROEPEN:
    ax.scatter(tb[g], ypos, color=colors_groep[g], s=60, label=g, zorder=2)
ax.set_yticks(ypos)
ax.set_yticklabels(tb["Domein"])
ax.set_xlabel("% verwacht dat het over 5 jaar slechter gaat (gewogen)")
ax.set_title("Waarover zijn groepen pessimistisch?", fontsize=14, fontweight="bold")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.22), ncol=2, fontsize=9)
ax.grid(axis="x", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("spoor1_pessimisme_per_domein.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 2c. Tevredenheid per domein per groep ───────────────────────────────────
rows = []
for d in domeinen_excl_leven:
    rows.append({"Domein": d, "Maat": "Gem. rapportcijfer", **per_groep(lambda m: wmean(tev.loc[m, d], w[m]))})
    rows.append({"Domein": d, "Maat": f"% onvoldoende (<{ONVOLDOENDE_GRENS})", **gw(onv[d])})

bewaar("2c_Tevredenheid_domein_groep", "Tevredenheid per levensdomein, per groep", VR2, pd.DataFrame(rows),
       "Gewogen. Per domein: gemiddeld cijfer en % onvoldoende.", visual="Heatmap of gegroepeerde staaf")

In [ ]:
tb = tabellen["2c_Tevredenheid_domein_groep"]["df"]
hm = tb[tb["Maat"] == "Gem. rapportcijfer"].set_index("Domein")[GROEPEN + [TOTAAL]]
hm = hm.sort_values(GROEPEN[3])

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(hm, annot=True, fmt=".1f", cmap="RdYlGn", vmin=4, vmax=9, linewidths=0.5, linecolor="white",
            cbar_kws={"label": "Gemiddeld rapportcijfer"}, ax=ax)
ax.set_xticklabels([c.replace(" & ", " &\n") for c in hm.columns], rotation=0, fontsize=9)
ax.set_ylabel("")
ax.set_title("Tevredenheid per levensdomein, per groep (gewogen)", fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig("spoor1_domeinscores_per_groep.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 2d. Invloed per domein per groep ────────────────────────────────────────
rows = []
for d in inv.columns:
    rows.append({"Domein": d, "Maat": "% veel invloed", **gw(ind(inv[d] == 2, inv[d]))})
    rows.append({"Domein": d, "Maat": "% geen invloed", **gw(ind(inv[d] == 0, inv[d]))})
for naam, c in stel_kort:
    rows.append({"Domein": f"Stelling: {naam}", "Maat": "% (helemaal) eens", **gw(ind(stel[c] >= 4, stel[c]))})

bewaar("2d_Invloed_per_groep", "Ervaren invloed (grip) per domein, per groep", VR2, pd.DataFrame(rows),
       "Gewogen. 'Weet ik niet' niet meegeteld.", visual="Gegroepeerde staaf")

In [ ]:
tb = tabellen["2d_Invloed_per_groep"]["df"]
sub = tb[tb["Maat"].isin(["% veel invloed", "% (helemaal) eens"])].copy()
sub["Label"] = np.where(sub["Maat"] == "% veel invloed", "Veel invloed: " + sub["Domein"], sub["Domein"])
sub = sub.set_index("Label")[GROEPEN[::-1]].iloc[::-1]   # groepen van boven naar beneden in vaste volgorde

fig, ax = plt.subplots(figsize=(10, 0.55 * len(sub) + 2))
sub.plot(kind="barh", ax=ax, color=[colors_groep[g] for g in GROEPEN[::-1]], width=0.8, edgecolor="white", legend=False)
ax.set_yticklabels([textwrap.fill(l, 55) for l in sub.index], fontsize=9)
ax.set_xlabel("% (gewogen)")
ax.set_ylabel("")
ax.set_title("Ervaren invloed en kansen, per groep", fontsize=14, fontweight="bold")
ax.legend([plt.Rectangle((0, 0), 1, 1, color=colors_groep[g]) for g in GROEPEN], GROEPEN,
          loc="lower center", bbox_to_anchor=(0.5, -0.12), ncol=2, fontsize=9)
ax.grid(axis="x", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("spoor1_invloed_per_groep.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 2e/2f. Wie zitten erin? Achtergrond en verleden ─────────────────────────
INDEX = "Index ontevreden&pess. (totaal=100)"

for naam_tab, titel, kenmerken, blokken in [
        ("2e_Profiel_achtergrond", "Wie zitten er in elke groep? Achtergrondkenmerken",
         achtergrond, [("Dagbesteding (meerdere mogelijk)", dagbesteding)]),
        ("2f_Profiel_verleden", "Wie zitten er in elke groep? Vroegere en recente (financiële) situatie",
         verleden, [])]:
    rows = [n_rij(Kenmerk="n (ongewogen)", Categorie="")]
    for naam, col in kenmerken:
        for cat in ordered_cats(df, col):
            rows.append({"Kenmerk": naam, "Categorie": cat, **gw(ind(df[col] == cat, df[col]))})
    for naam, blok in blokken:
        for c in blok.columns:
            rows.append({"Kenmerk": naam, "Categorie": c, **gw(blok[c])})
    tb = pd.DataFrame(rows)
    tb[INDEX] = 100 * tb[GROEPEN[3]] / tb[TOTAAL]
    tb.loc[0, INDEX] = np.nan
    bewaar(naam_tab, titel, VR2, tb,
           "Kolom-percentages: verdeling van elk kenmerk BINNEN de groep. Index > 100 = oververtegenwoordigd in "
           "'Ontevreden & pessimistisch' t.o.v. totaal (150 = 1,5x zo vaak).",
           fmt={INDEX: "0"}, visual="Profielkaart per groep / gegroepeerde staaf")

In [ ]:
# ── 2g. Onzekerheden, netwerk en hulp per groep ─────────────────────────────
rows = []
for thema, blok in [("Financiële/bestaansonzekerheid", onz_fin), ("Overige onzekerheid", onz_ovr),
                    ("Iemand die kan helpen met", net), ("Weet waar hulp te vinden bij", hulp)]:
    for c in blok.columns:
        rows.append({"Thema": thema, "Item": c, **gw(blok[c])})
tb = pd.DataFrame(rows)
tb[INDEX] = 100 * tb[GROEPEN[3]] / tb[TOTAAL]

bewaar("2g_Onzekerheid_netwerk_groep", "Onzekerheden, sociaal netwerk en hulp per groep", VR2, tb,
       "Gewogen % 'van toepassing' / 'ja'. Bij netwerk en hulp telt 'weet ik niet' als 'nee' (geen zekere steun).",
       fmt={INDEX: "0"}, visual="Gegroepeerde staaf")

In [ ]:
# ── Figuren: wie zitten er in elke groep? ───────────────────────────────────
def plot_profiel(tb, titel, bestand, ncols=3, horizontaal=False):
    kenmerken = [k for k in tb["Kenmerk"].unique() if k != "n (ongewogen)"]
    nrows = int(np.ceil(len(kenmerken) / ncols))
    hoogte = [max(3.8, 0.45 * (tb["Kenmerk"] == k).sum() + 1.5) for k in kenmerken] if horizontaal else [3.8]
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.5 * ncols, max(hoogte) * nrows), squeeze=False)
    axes = axes.flatten()
    for ax, k in zip(axes, kenmerken):
        volgorde = GROEPEN[::-1] if horizontaal else GROEPEN   # barh tekent van onder naar boven
        sub = tb[tb["Kenmerk"] == k].set_index("Categorie")[volgorde]
        if horizontaal:
            sub = sub.iloc[::-1]
        sub.plot(kind="barh" if horizontaal else "bar", ax=ax, color=[colors_groep[g] for g in volgorde],
                 width=0.8, edgecolor="white", legend=False)
        labels = [textwrap.fill(str(c), 40 if horizontaal else 16) for c in sub.index]
        if horizontaal:
            ax.set_yticklabels(labels, fontsize=8)
            ax.set_xlabel("% binnen groep")
            ax.set_ylabel("")
            ax.grid(axis="x", alpha=0.3)
        else:
            ax.set_xticklabels(labels, rotation=0, fontsize=8)
            ax.set_ylabel("% binnen groep")
            ax.set_xlabel("")
            ax.grid(axis="y", alpha=0.3)
        ax.set_title(k, fontweight="bold")
        ax.spines[["top", "right"]].set_visible(False)
    for ax in axes[len(kenmerken):]:
        ax.axis("off")
    handles = [plt.Rectangle((0, 0), 1, 1, color=colors_groep[g]) for g in GROEPEN]
    fig.legend(handles, GROEPEN, loc="lower center", ncol=4, fontsize=10, bbox_to_anchor=(0.5, -0.01))
    fig.suptitle(titel, fontsize=16, fontweight="bold")
    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    plt.savefig(bestand, dpi=150, bbox_inches="tight")
    plt.show()

plot_profiel(tabellen["2e_Profiel_achtergrond"]["df"], "Wie zitten er in elke groep? Achtergrond (gewogen)",
             "spoor1_profiel_achtergrond.png")
plot_profiel(tabellen["2f_Profiel_verleden"]["df"], "Wie zitten er in elke groep? Jeugd en financiële situatie (gewogen)",
             "spoor1_profiel_verleden.png", ncols=2)
plot_profiel(tabellen["2g_Onzekerheid_netwerk_groep"]["df"].rename(columns={"Thema": "Kenmerk", "Item": "Categorie"}),
             "Onzekerheden, netwerk en hulp per groep (gewogen)", "spoor1_profiel_onzekerheid_netwerk.png",
             ncols=2, horizontaal=True)

In [ ]:
# ── Figuur: wat kenmerkt 'Ontevreden & pessimistisch'? (index t.o.v. totaal) ─
prof = pd.concat([tabellen["2e_Profiel_achtergrond"]["df"],
                  tabellen["2f_Profiel_verleden"]["df"],
                  tabellen["2g_Onzekerheid_netwerk_groep"]["df"].rename(columns={"Thema": "Kenmerk", "Item": "Categorie"})])
prof = prof[(prof["Kenmerk"] != "n (ongewogen)") & (prof[TOTAAL] >= 5)].copy()   # kleine categorieën weglaten
prof = prof[~prof["Categorie"].astype(str).str.lower().str.contains("wil ik niet|weet ik niet|anders")]
prof["Label"] = prof["Kenmerk"] + ": " + prof["Categorie"].astype(str)
prof["afwijking"] = prof[INDEX] - 100
top = pd.concat([prof.nlargest(12, "afwijking"), prof.nsmallest(6, "afwijking")]).sort_values("afwijking")

fig, ax = plt.subplots(figsize=(11, 0.45 * len(top) + 2))
ax.barh([textwrap.fill(l, 70) for l in top["Label"]], top["afwijking"],
        color=np.where(top["afwijking"] > 0, colors_groep[GROEPEN[3]], "#95a5a6"))
for i, (a, idx) in enumerate(zip(top["afwijking"], top[INDEX])):   # index-getal net naast de nullijn
    ax.text(-0.5 if a > 0 else 0.5, i, f"{idx:.0f}", va="center", ha="right" if a > 0 else "left",
            fontsize=8, color="#555555")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Index t.o.v. totaal - 100  (rechts = vaker in 'Ontevreden & pessimistisch', getal = index)")
ax.set_title("Wat kenmerkt de groep 'Ontevreden & pessimistisch'?", fontsize=14, fontweight="bold")
ax.tick_params(axis="y", labelsize=8)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("spoor1_kenmerken_ontevreden_pessimistisch.png", dpi=150, bbox_inches="tight")
plt.show()

## Subvraag 3 en 4 – Basisdingen niet op orde → meer pessimisme? Wat zijn basisdingen?

In [ ]:
# ── 3a. Pessimisme naar aantal domeinen met onvoldoende ─────────────────────
VR3 = "3. Is de kans op pessimisme groter als basisdingen (domeinen) niet op orde zijn?"
nb = df["_n_onvoldoende"].clip(upper=4)
rows = []
for k, lab in enumerate(["0", "1", "2", "3", "4 of meer"]):
    m = nb == k
    rows.append({"Aantal domeinen met onvoldoende": lab,
                 "% van de bevolking": wpct(ind(m, nb), w),
                 "% pessimistisch over leven": wpct(df.loc[m, "_pessimistisch"], w[m]),
                 f"% tevreden met leven ({TEVREDEN_GRENS}+)": wpct(df.loc[m, "_tevreden"], w[m]),
                 "Gem. tevredenheid leven": wmean(df.loc[m, "_tev_leven"], w[m]),
                 "% Ontevreden & pessimistisch": wpct(ind(df.loc[m, "_groep"] == GROEPEN[3], df.loc[m, "_groep"]), w[m]),
                 "n": int(m.sum())})
tb_3a = pd.DataFrame(rows)

bewaar("3a_Pessimisme_x_aantal_domeinen", "Hoe meer domeinen niet op orde, hoe pessimistischer?", VR3, tb_3a,
       f"Aantal van de {onv.shape[1]} levensdomeinen (excl. 'leven in het algemeen') met een onvoldoende (cijfer <{ONVOLDOENDE_GRENS}).",
       fmt={"Gem. tevredenheid leven": "0.0"}, visual="Oplopende staaf: % pessimistisch")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(tb_3a["Aantal domeinen met onvoldoende"], tb_3a["% pessimistisch over leven"], color="#e74c3c")
for i, v in enumerate(tb_3a["% pessimistisch over leven"]):
    ax.text(i, v + 1, f"{v:.0f}%", ha="center", fontsize=10)
ax.set_xlabel(f"Aantal levensdomeinen met een onvoldoende (<{ONVOLDOENDE_GRENS})")
ax.set_ylabel("% pessimistisch over het leven")
ax.set_title("Hoe meer domeinen niet op orde, hoe pessimistischer", fontsize=14, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("spoor1_pessimisme_aantal_domeinen.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 3b. Kans op pessimisme per domein (onvoldoende vs voldoende) ────────────
rows = []
for d in onv.columns:
    m1, m0 = onv[d] == 1, onv[d] == 0
    p1, p0 = wpct(df.loc[m1, "_pessimistisch"], w[m1]), wpct(df.loc[m0, "_pessimistisch"], w[m0])
    mdl, n = fit_logit(df["_pessimistisch"], pd.concat([onv[[d]], Cs], axis=1), w)
    lo, hi = np.exp(mdl.conf_int().loc[d])
    rows.append({"Domein": d,
                 f"% met onvoldoende (<{ONVOLDOENDE_GRENS})": wpct(onv[d], w),
                 "% pessimistisch als domein onvoldoende": p1,
                 "% pessimistisch als domein voldoende": p0,
                 "Verschil (%-punt)": p1 - p0,
                 "Relatieve kans (x zo vaak)": p1 / p0 if p0 else np.nan,
                 "Odds ratio (gecorrigeerd)": np.exp(mdl.params[d]), "OR 95%-BI onder": lo, "OR 95%-BI boven": hi,
                 "p": mdl.pvalues[d], "n onvoldoende": int(m1.sum())})
tb = pd.DataFrame(rows).sort_values("Verschil (%-punt)", ascending=False)

bewaar("3b_Pessimisme_x_domein", "Kans op pessimisme over het leven als een domein niet op orde is", VR3, tb,
       "Odds ratio uit gewogen logistische regressie, per domein apart, gecorrigeerd voor leeftijd, geslacht, opleiding, "
       "inkomen (OR > 1 = grotere kans op pessimisme).",
       fmt={"Relatieve kans (x zo vaak)": "0.0", "Odds ratio (gecorrigeerd)": "0.00", "OR 95%-BI onder": "0.00",
            "OR 95%-BI boven": "0.00", "p": "0.000"},
       visual="Gesorteerde staaf (verschil in %-punt)")

In [ ]:
# ── 4. Basisdingen: alle domeinen tegelijk in één model ─────────────────────
VR4 = "4. Wat zijn 'basisdingen'? Welke domeinen wegen het zwaarst?"
Xd = pd.DataFrame({d: z(tev[d]) for d in domeinen_excl_leven})
m_tev, n_tev = fit_wls(z(df["_tev_leven"]), pd.concat([Xd, Cs], axis=1), w)
m_pes, n_pes = fit_logit(df["_pessimistisch"], pd.concat([onv, Cs], axis=1), w)

tb = pd.DataFrame([{"Domein": d,
                    "Beta op tevredenheid leven (samen)": m_tev.params[d],
                    "p (tevredenheid)": m_tev.pvalues[d],
                    "Odds ratio pessimisme bij onvoldoende (samen)": np.exp(m_pes.params[d]),
                    "p (pessimisme)": m_pes.pvalues[d],
                    f"% met onvoldoende (<{ONVOLDOENDE_GRENS})": wpct(onv[d], w),
                    "Gem. rapportcijfer": wmean(tev[d], w)} for d in domeinen_excl_leven])
tb["Rang tevredenheid"] = tb["Beta op tevredenheid leven (samen)"].rank(ascending=False)
tb["Rang pessimisme"] = tb["Odds ratio pessimisme bij onvoldoende (samen)"].rank(ascending=False)
tb["Gemiddelde rang"] = tb[["Rang tevredenheid", "Rang pessimisme"]].mean(axis=1)
tb = tb.sort_values("Gemiddelde rang")

bewaar("4_Basisdingen", "Welke levensdomeinen zijn 'basisdingen'? (alle domeinen tegelijk in één model)", VR4, tb,
       f"Overlap tussen domeinen weggefilterd. Beta: effect domein-tevredenheid op tevredenheid met het leven (WLS, n={n_tev}). "
       f"Odds ratio: kans op pessimisme bij een onvoldoende op dat domein (logistisch, n={n_pes}). Gecorrigeerd voor leeftijd, "
       "geslacht, opleiding, inkomen. Laagste gemiddelde rang = meest 'basaal'.",
       fmt={"Beta op tevredenheid leven (samen)": "0.00", "p (tevredenheid)": "0.000",
            "Odds ratio pessimisme bij onvoldoende (samen)": "0.00", "p (pessimisme)": "0.000",
            "Gem. rapportcijfer": "0.0", "Rang tevredenheid": "0", "Rang pessimisme": "0", "Gemiddelde rang": "0.0"},
       visual="Top-5 lijst / gesorteerde staaf")

## Subvraag 5 – Zekerheid nu vs. zekerheid in de toekomst

In [ ]:
# ── 5a. Per domein: nu vs toekomst ──────────────────────────────────────────
VR5 = "5. Is zekerheid NU anders dan zekerheid over de TOEKOMST?"
rows = []
for d in domeinen:
    s = tev[d]
    row = {"Domein": d,
           "Gem. rapportcijfer nu": wmean(s, w),
           f"% tevreden nu ({TEVREDEN_GRENS}+)": wpct(ind(s >= TEVREDEN_GRENS, s), w),
           f"% onvoldoende nu (<{ONVOLDOENDE_GRENS})": wpct(ind(s < ONVOLDOENDE_GRENS, s), w)}
    if d in toe.columns:
        t = toe[d]
        row["% verwacht beter"] = wpct(ind(t > 3, t), w)
        row["% verwacht hetzelfde"] = wpct(ind(t == 3, t), w)
        row["% verwacht slechter"] = wpct(ind(t < 3, t), w)
        row["Saldo toekomst (beter - slechter)"] = row["% verwacht beter"] - row["% verwacht slechter"]
        m = s >= TEVREDEN_GRENS
        row["% van tevredenen nu dat slechter verwacht"] = wpct(ind(t[m] < 3, t[m]), w[m])
    rows.append(row)

bewaar("5a_Nu_vs_toekomst_per_domein", "Hoe staat het er nu voor en hoe kijkt men vooruit, per domein", VR5, pd.DataFrame(rows),
       "Gewogen. Saldo > 0 = meer verwachte verbetering dan verslechtering. Laatste kolom: 'nu zeker, later onzeker'.",
       fmt={"Gem. rapportcijfer nu": "0.0"}, visual="Dumbbell: % tevreden nu vs % verwacht slechter")

In [ ]:
# ── 5b. Onzekerheden: meer verbonden met nu of met de toekomst? ─────────────
items = pd.concat([onz_fin, onz_ovr], axis=1)
thema = {**{c: "Financieel/bestaan" for c in onz_fin.columns}, **{c: "Overig" for c in onz_ovr.columns}}
m_nu, _  = fit_wls(z(df["_tev_leven"]), pd.concat([items, Cs], axis=1), w)
m_toe, _ = fit_wls(z(df["_toe_leven"]), pd.concat([items, Cs], axis=1), w)

rows = []
for c in items.columns:
    m1, m0 = items[c] == 1, items[c] == 0
    b_nu, b_toe = m_nu.params[c], m_toe.params[c]
    rows.append({"Thema": thema[c], "Onzekerheid": c,
                 "% van toepassing": wpct(items[c], w),
                 "Gem. tevredenheid leven - wel": wmean(df.loc[m1, "_tev_leven"], w[m1]),
                 "Gem. tevredenheid leven - niet": wmean(df.loc[m0, "_tev_leven"], w[m0]),
                 "% pessimistisch - wel": wpct(df.loc[m1, "_pessimistisch"], w[m1]),
                 "% pessimistisch - niet": wpct(df.loc[m0, "_pessimistisch"], w[m0]),
                 "Effect op tevredenheid NU (SD)": b_nu, "p (nu)": m_nu.pvalues[c],
                 "Effect op verwachting TOEKOMST (SD)": b_toe, "p (toekomst)": m_toe.pvalues[c],
                 "Sterkst verbonden met": "nu" if abs(b_nu) > abs(b_toe) else "toekomst",
                 "n wel": int(m1.sum())})

bewaar("5b_Onzekerheid_nu_vs_toekomst", "Hangen onzekerheden meer samen met tevredenheid nu of met verwachting voor de toekomst?",
       VR5, pd.DataFrame(rows),
       "Links beschrijvend (wel/niet van toepassing). Effect-kolommen: alle onzekerheden tegelijk in één gewogen regressie, "
       "gecorrigeerd voor leeftijd, geslacht, opleiding, inkomen; uitkomsten gestandaardiseerd zodat nu (1-10) en toekomst "
       "(1-5) vergelijkbaar zijn. Negatief = lager cijfer / somberder verwachting.",
       fmt={"Gem. tevredenheid leven - wel": "0.0", "Gem. tevredenheid leven - niet": "0.0",
            "Effect op tevredenheid NU (SD)": "0.00", "Effect op verwachting TOEKOMST (SD)": "0.00",
            "p (nu)": "0.000", "p (toekomst)": "0.000"},
       visual="Twee staven per item (nu vs toekomst)")

In [ ]:
# ── 5c. Opeenstapeling financiële onzekerheid ───────────────────────────────
nb = df["_n_onz_fin"].clip(upper=3)
rows = []
for k, lab in enumerate(["0", "1", "2", "3 of meer"]):
    m = nb == k
    rows.append({"Aantal financiële onzekerheden": lab,
                 "% van de bevolking": wpct(ind(m, nb), w),
                 "Gem. tevredenheid leven (nu)": wmean(df.loc[m, "_tev_leven"], w[m]),
                 f"% tevreden nu ({TEVREDEN_GRENS}+)": wpct(df.loc[m, "_tevreden"], w[m]),
                 "Gem. toekomstverwachting leven (1-5)": wmean(df.loc[m, "_toe_leven"], w[m]),
                 "% pessimistisch over leven": wpct(df.loc[m, "_pessimistisch"], w[m]),
                 "% veel invloed op leven": wpct(df.loc[m, "_veel_invloed"], w[m]),
                 "n": int(m.sum())})

bewaar("5c_Stapeling_onzekerheid", "Opeenstapeling van financiële onzekerheid: nu en toekomst", VR5, pd.DataFrame(rows),
       "Aantal aangevinkte financiële/bestaansonzekerheden (schulden, geen vast inkomen, moeite met rondkomen, enz.).",
       fmt={"Gem. tevredenheid leven (nu)": "0.0", "Gem. toekomstverwachting leven (1-5)": "0.00"},
       visual="Twee lijnen: % tevreden nu en % pessimistisch")

## 6 – Invloed / grip

In [ ]:
# ── 6a. Invloed per domein ──────────────────────────────────────────────────
VR6 = "6. Gevoel van invloed (grip): hoe hangt het samen met tevredenheid en toekomstperspectief?"
rows = []
for d in inv.columns:
    s = inv[d]
    row = {"Domein": d}
    for k, lab in INVLOED_LABELS.items():
        row[f"% {lab.lower()}"] = wpct(ind(s == k, s), w)
    ruw = norm(df[inv_raw[d]])
    row["% weet ik niet (van alle antwoorden)"] = wpct(ind(ruw.str.contains("weet", na=False), ruw), w)
    for k in (0, 2):
        m = s == k
        if d in tev.columns:
            row[f"Gem. cijfer domein - {INVLOED_LABELS[k].lower()}"] = wmean(tev.loc[m, d], w[m])
        if d in toe.columns:
            row[f"% verwacht slechter - {INVLOED_LABELS[k].lower()}"] = wpct(ind(toe.loc[m, d] < 3, toe.loc[m, d]), w[m])
    row["r invloed x tevredenheid"] = wcorr(s, tev[d], w)[0] if d in tev.columns else np.nan
    row["r invloed x toekomstverwachting"] = wcorr(s, toe[d], w)[0] if d in toe.columns else np.nan
    rows.append(row)
tb = pd.DataFrame(rows)

bewaar("6a_Invloed_per_domein", "Hoeveel invloed ervaren mensen per levensdomein, en maakt dat verschil?", VR6, tb,
       "Verdeling exclusief 'weet ik niet'. Daarna: gemiddeld cijfer en % dat verslechtering verwacht, voor wie geen vs. veel "
       "invloed ervaart op dat domein. r = gewogen correlatie.",
       fmt={c: ("0.00" if c.startswith("r ") else "0.0") for c in tb.columns if c != "Domein"},
       visual="100%-gestapelde staaf invloed per domein + verschilstaaf")

In [ ]:
# ── 6b. 'Vastzitten': onvoldoende én geen invloed ───────────────────────────
VAST = "% onvoldoende én geen invloed ('vastzitten')"
rows = []
for d in [d for d in inv.columns if d in tev.columns]:
    s, t = inv[d], tev[d]
    ok, laag = s.notna() & t.notna(), t < ONVOLDOENDE_GRENS
    rows.append({"Domein": d,
                 VAST: wpct(ind(laag & (s == 0), s.where(ok)), w),
                 "% onvoldoende, wel (enige) invloed": wpct(ind(laag & (s >= 1), s.where(ok)), w),
                 "% van wie onvoldoende geeft: geen invloed": wpct((s[laag & ok] == 0).astype(float), w[laag & ok]),
                 "n onvoldoende": int((laag & ok).sum())})
tb_6b = pd.DataFrame(rows).sort_values(VAST, ascending=False)

bewaar("6b_Vastzitten_per_domein", "Waar zitten mensen 'vast'? Ontevreden én geen gevoel van invloed", VR6, tb_6b,
       f"'Vastzitten' = onvoldoende (<{ONVOLDOENDE_GRENS}) voor een domein én er geen invloed op ervaren. "
       "Daar ligt een rol voor beleid: mensen kunnen het niet zelf oplossen.", visual="Gesorteerde staaf")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
tb_plot = tb_6b.sort_values(VAST)
ax.barh(tb_plot["Domein"], tb_plot[VAST], color="#e74c3c", label="Onvoldoende én geen invloed")
ax.barh(tb_plot["Domein"], tb_plot["% onvoldoende, wel (enige) invloed"], left=tb_plot[VAST],
        color="#f5b7b1", label="Onvoldoende, wel (enige) invloed")
ax.set_xlabel("% van de bevolking (gewogen)")
ax.set_title("Waar zitten mensen 'vast'?", fontsize=14, fontweight="bold")
ax.legend(loc="lower right", fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("spoor1_vastzitten.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 6c. Buffer: beschermen grip of netwerk tegen pessimisme als het tegenzit? ─
contexten = [("1+ financiële onzekerheid", df["_n_onz_fin"] >= 1),
             ("Geen financiële onzekerheid", df["_n_onz_fin"] == 0),
             ("2+ domeinen onvoldoende", df["_n_onvoldoende"] >= 2),
             ("0-1 domeinen onvoldoende", df["_n_onvoldoende"] <= 1)]
beschermers = [("Ervaren invloed op leven", df["_inv_leven"], INVLOED_LABELS)]
for key in ("Geld aan u te lenen", "luisterend oor"):
    c = next((c for c in net.columns if key.lower() in c.lower()), None)
    if c:
        beschermers.append((f"Iemand die kan helpen: {c}", net[c], {0: "Nee/weet niet", 1: "Ja"}))

rows = []
for ctx, cm in contexten:
    for bnaam, b, labels in beschermers:
        for k, lab in labels.items():
            m = cm & (b == k)
            rows.append({"Situatie": ctx, "Beschermende factor": bnaam, "Categorie": lab,
                         "% pessimistisch over leven": wpct(df.loc[m, "_pessimistisch"], w[m]),
                         "Gem. tevredenheid leven": wmean(df.loc[m, "_tev_leven"], w[m]),
                         f"% tevreden ({TEVREDEN_GRENS}+)": wpct(df.loc[m, "_tevreden"], w[m]),
                         "n": int(m.sum())})

# toets: beschermt veel invloed extra bij tegenslag? (interactie, OR < 1 = extra bescherming)
toets = []
for naam, risico in [("financiële onzekerheid", ind(df["_n_onz_fin"] >= 1, df["_n_onz_fin"])),
                     ("2+ domeinen onvoldoende", ind(df["_n_onvoldoende"] >= 2, df["_n_onvoldoende"]))]:
    X = pd.concat([risico.rename("risico"), df["_veel_invloed"].rename("veel_invloed"),
                   (risico * df["_veel_invloed"]).rename("interactie"), Cs], axis=1)
    mdl, _ = fit_logit(df["_pessimistisch"], X, w)
    toets.append(f"{naam}: interactie OR={np.exp(mdl.params['interactie']):.2f}, p={mdl.pvalues['interactie']:.3f}")
print("=== Buffer-toetsen ===")
print("\n".join(toets))

bewaar("6c_Buffer_invloed_netwerk", "Beschermt grip of een sociaal netwerk tegen pessimisme als het tegenzit?", VR6,
       pd.DataFrame(rows),
       "Vergelijk binnen dezelfde situatie wie wel/geen invloed of steun ervaart. Toets of veel invloed extra beschermt bij "
       "tegenslag (logistische regressie, interactie; OR < 1 = extra bescherming): " + "; ".join(toets),
       fmt={"Gem. tevredenheid leven": "0.0"}, visual="Gegroepeerde staaf per situatie")

In [ ]:
# ── 6d. Regressie toekomstverwachting: telt grip mee bovenop tevredenheid nu? ─
preds = {"Tevredenheid leven nu (1-10)": df["_tev_leven"],
         "Ervaren invloed op leven (0-2)": df["_inv_leven"]}
for naam, c in stel_kort[:2]:
    preds[f"Stelling: {naam} (1-5)"] = stel[c]

idx = pd.concat([df["_toe_leven"]] + list(preds.values()) + [C, w], axis=1).dropna().index
y   = z(df.loc[idx, "_toe_leven"])
Xp  = pd.DataFrame({k: z(s.loc[idx]) for k, s in preds.items()})
Cz  = C.loc[idx].copy()
Cz["Leeftijd"] = z(Cz["Leeftijd"])

modellen = {"Model 1: achtergrond + tevredenheid nu": list(preds)[:1],
            "Model 2: + invloed/grip": list(preds)}
res = {naam: fit_wls(y, pd.concat([Cz, Xp[p]], axis=1), w.loc[idx]) for naam, p in modellen.items()}

rows = []
for k in list(preds) + ["Leeftijd"]:
    row = {"Voorspeller": k}
    for naam, (mdl, n) in res.items():
        row[f"{naam} - beta"] = mdl.params.get(k, np.nan)
        row[f"{naam} - p"] = mdl.pvalues.get(k, np.nan)
    rows.append(row)
rows.append({"Voorspeller": "R² (verklaarde variantie)", **{f"{naam} - beta": mdl.rsquared for naam, (mdl, n) in res.items()}})
rows.append({"Voorspeller": "n", **{f"{naam} - beta": n for naam, (mdl, n) in res.items()}})
tb = pd.DataFrame(rows)

bewaar("6d_Regressie_toekomst", "Regressie: wat voorspelt de toekomstverwachting? Telt grip mee bovenop tevredenheid nu?",
       VR6, tb,
       "Gewogen lineaire regressie, gestandaardiseerde beta's. Uitkomst: verwachting voor het leven over 5 jaar (1-5). "
       "Gecorrigeerd voor leeftijd, geslacht, opleiding, inkomen. Toename R² = extra verklaring door grip.",
       fmt={c: ("0.000" if c.endswith("- p") else "0.00") for c in tb.columns if c != "Voorspeller"},
       visual="Kernboodschap")

## 7 – Extra: kansenkaart en redenen achter pessimisme/optimisme

In [ ]:
# ── 7a. Uitkomsten naar achtergrond en verleden ('kansenkaart') ─────────────
uitkomsten = {
    "Gem. tevredenheid leven (1-10)": lambda m: wmean(df.loc[m, "_tev_leven"], w[m]),
    f"% tevreden ({TEVREDEN_GRENS}+)": lambda m: wpct(df.loc[m, "_tevreden"], w[m]),
    "% pessimistisch over leven": lambda m: wpct(df.loc[m, "_pessimistisch"], w[m]),
    "% optimistisch over leven": lambda m: wpct(df.loc[m, "_optimistisch"], w[m]),
    "% veel invloed op leven": lambda m: wpct(df.loc[m, "_veel_invloed"], w[m]),
    "% Ontevreden & pessimistisch": lambda m: wpct(ind(df.loc[m, "_groep"] == GROEPEN[3], df.loc[m, "_groep"]), w[m]),
}
for naam, c in stel_kort:
    uitkomsten[f"% eens: {naam}"] = lambda m, c=c: wpct(ind(stel.loc[m, c] >= 4, stel.loc[m, c]), w[m])

alle = df["_tev_leven"].notna()
rows = [{"Kenmerk": "Totaal", "Categorie": "", **{k: f(alle) for k, f in uitkomsten.items()}, "n": int(alle.sum())}]
for naam, col in achtergrond + verleden:
    for cat in ordered_cats(df, col):
        m = df[col] == cat
        rows.append({"Kenmerk": naam, "Categorie": cat, **{k: f(m) for k, f in uitkomsten.items()}, "n": int(m.sum())})
for c in dagbesteding.columns:
    m = dagbesteding[c] == 1
    rows.append({"Kenmerk": "Dagbesteding", "Categorie": c, **{k: f(m) for k, f in uitkomsten.items()}, "n": int(m.sum())})

bewaar("7a_Uitkomsten_naar_kenmerken", "Tevredenheid, toekomstblik en grip naar achtergrond en verleden",
       "Extra: welke groepen scoren laag? (kansengelijkheid: jeugd -> nu)", pd.DataFrame(rows),
       "Gewogen. Rij = groep mensen, kolom = uitkomst. Vergelijk met 'Totaal'. Let op kleine n.",
       fmt={"Gem. tevredenheid leven (1-10)": "0.0"}, visual="Heatmap / 'kansenkaart'")

In [ ]:
# ── 7b/7c. Redenen achter pessimisme en optimisme ───────────────────────────
# twee vragenlijstversies ('hetzelfde of slechter' en 'slechter') samengevoegd op antwoordoptie
themas = [("Lichamelijke gezondheid", "lichamelijke gezondheid"), ("Woning", "woonsituatie"),
          ("Sociale contacten", "sociale contacten")]

for richting, naam_tab, titel, uitleg in [
        ("slechter", "7b_Redenen_pessimisme", "Waarom verwachten mensen dat het slechter gaat? (gezondheid, wonen, sociaal)",
         "Onder wie verwacht dat het domein slechter wordt én een reden gaf. Meerdere redenen mogelijk (telt niet op tot 100%)."),
        ("beter", "7c_Redenen_optimisme", "Waarom verwachten mensen dat het beter gaat? (gezondheid, wonen, sociaal)",
         "Onder wie verwacht dat het domein beter wordt én een reden gaf. Meerdere redenen mogelijk.")]:
    rows = []
    for dom, key in themas:
        cols = [c for c in find_cols(df, "u denkt dat uw " + key, "redenen")
                if ("slechter" in clean_label(c).lower()) == (richting == "slechter")
                and "weet ik niet" not in subject(c).lower()]
        if not cols or dom not in toe.columns:
            continue
        blok = binary_block(df, cols)
        opties = {}
        for c in cols:
            opties.setdefault(re.sub(r"[\s:…\.]+$", "", subject(c)), []).append(c)
        sel = pd.DataFrame({o: blok[cs].max(axis=1) for o, cs in opties.items()})
        doel = (toe[dom] < 3) if richting == "slechter" else (toe[dom] > 3)
        basis = doel & (sel.fillna(0).sum(axis=1) > 0)
        subsets = [("Iedereen", basis)] + [(g, basis & (df["_groep"] == g)) for g in GROEPEN] + \
                  [(lg, basis & (df["Leeftijdsgroep"] == lg)) for lg in (LEEFTIJD_LABELS[0], LEEFTIJD_LABELS[-1])]
        for o in sel.columns:
            rows.append({"Domein": dom, "Reden": o, **{s: wpct(sel.loc[m, o], w[m]) for s, m in subsets}})
        rows.append({"Domein": dom, "Reden": "n (ongewogen)", **{s: int(m.sum()) for s, m in subsets}})
    if rows:
        bewaar(naam_tab, titel, "Extra: achtergrond bij subvraag 2 (waarover en waarom pessimistisch)",
               pd.DataFrame(rows), uitleg, visual="Horizontale staaf per domein")
    else:
        print(f"LET OP: geen redenkolommen gevonden voor {naam_tab}")

## 8 – Extra: bereik van hulp en doorwerking van de jeugd

In [ ]:
# ── 8a. Weten mensen mét een probleem waar ze hulp kunnen vinden? ───────────
def kolom(blok, key):
    return next((c for c in blok.columns if key.lower() in c.lower()), None)

paren = [  # (probleem, 0/1-reeks, hulp-item)
    ("Heeft schulden",                      onz_fin.get(kolom(onz_fin, "schulden")),                 "geldhulp"),
    ("Heeft moeite om rond te komen",       onz_fin.get(kolom(onz_fin, "moeite om rond te komen")),  "geldhulp"),
    ("Woonsituatie is onzeker",             onz_fin.get(kolom(onz_fin, "woonsituatie is onzeker")),  "(nieuwe) woning"),
    ("Mentale klachten die onzeker maken",  onz_ovr.get(kolom(onz_ovr, "mentale klachten")),         "mentale problemen"),
    (f"Psychische gezondheid onvoldoende (<{ONVOLDOENDE_GRENS})", onv.get("Psychische gezondheid"),  "mentale problemen"),
    ("Gezondheid zorgt voor onzekerheid",   onz_ovr.get(kolom(onz_ovr, "gezondheid zorgt")),         "gezondheidszorg"),
    ("Is werkzoekend",                      dagbesteding.get(kolom(dagbesteding, "werkzoekend")),    "baan of vrijwilligerswerk"),
]

rows = []
for probleem, s, key in paren:
    h = kolom(hulp, key)
    if s is None or h is None:
        print(f"LET OP: overgeslagen ({probleem} / {key})")
        continue
    m = s == 1
    weet, weet_niet = m & (hulp[h] == 1), m & (hulp[h] == 0)
    rows.append({"Probleem": probleem, "Hulp bij": h,
                 "% van bevolking met probleem": wpct(s, w),
                 "Van hen: % weet waar hulp te vinden": wpct(hulp.loc[m, h], w[m]),
                 "Van hen: % weet het niet": 100 - wpct(hulp.loc[m, h], w[m]),
                 "Hele bevolking: % weet waar hulp te vinden": wpct(hulp[h], w),
                 "% pessimistisch - weet waar hulp": wpct(df.loc[weet, "_pessimistisch"], w[weet]),
                 "% pessimistisch - weet het niet": wpct(df.loc[weet_niet, "_pessimistisch"], w[weet_niet]),
                 "n met probleem": int(m.sum())})
tb_8a = pd.DataFrame(rows)

bewaar("8a_Bereik_hulp", "Weten mensen met een probleem waar ze hulp kunnen vinden?",
       "Extra: bereik van hulp (relevant voor 'welke groepen kunnen we sturen')", tb_8a,
       "Gewogen. 'Weet het niet' = antwoord nee of weet ik niet op 'Ik weet waar of bij welke organisaties ik terecht kan'. "
       "Laatste kolommen: is wie de weg naar hulp kent minder pessimistisch?",
       visual="Staaf: % met probleem dat de weg naar hulp niet kent")

In [ ]:
if len(tb_8a):
    tb = tb_8a.sort_values("Van hen: % weet het niet")
    fig, ax = plt.subplots(figsize=(10, 0.6 * len(tb) + 2))
    ax.barh([f"{p}\n(hulp bij: {h.lower()})" for p, h in zip(tb["Probleem"], tb["Hulp bij"])],
            tb["Van hen: % weet het niet"], color="#e74c3c")
    for i, v in enumerate(tb["Van hen: % weet het niet"]):
        ax.text(v + 0.5, i, f"{v:.0f}%", va="center", fontsize=9)
    ax.set_xlabel("% dat NIET weet waar hulp te vinden is (gewogen)")
    ax.set_title("Mensen met een probleem die de weg naar hulp niet kennen", fontsize=14, fontweight="bold")
    ax.tick_params(axis="y", labelsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig("spoor1_bereik_hulp.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ── 8b. Doorwerking jeugd: hoeveel loopt via de huidige situatie? ───────────
# Model 1: jeugd + leeftijd/geslacht. Model 2: + opleiding, inkomen, financiële onzekerheid, domeinen onvoldoende.
# Daalt het effect van de jeugd in model 2, dan loopt dat deel via de huidige (financiële) situatie.
basis = [c for c in ["Leeftijd", "Vrouw"] if c in Cs.columns]
huidig = pd.concat([Cs.drop(columns=basis),
                    z(df["_n_onz_fin"]).rename("Aantal financiële onzekerheden"),
                    z(df["_n_onvoldoende"]).rename("Aantal domeinen onvoldoende")], axis=1)

rows = []
for naam, col in verleden:
    if "12 maanden" in naam:   # is huidige situatie, geen jeugd/verleden
        continue
    s = df[col].astype("object")
    s = s.where(~s.astype(str).str.lower().str.contains("weet|wil ik niet", na=False))
    cats = [c for c in ordered_cats(df, col) if c in set(s.dropna())]
    ref = cats[0]
    D = pd.get_dummies(s, dtype=float)[cats[1:]]
    D[s.isna()] = np.nan
    D.columns = [f"cat_{i}" for i in range(len(cats) - 1)]
    idx = pd.concat([D, Cs[basis], huidig, df[["_pessimistisch", "_tev_leven"]], w], axis=1).dropna().index

    m1p, _ = fit_logit(df.loc[idx, "_pessimistisch"], pd.concat([D, Cs[basis]], axis=1).loc[idx], w.loc[idx])
    m2p, n = fit_logit(df.loc[idx, "_pessimistisch"], pd.concat([D, Cs[basis], huidig], axis=1).loc[idx], w.loc[idx])
    m1t, _ = fit_wls(df.loc[idx, "_tev_leven"], pd.concat([D, Cs[basis]], axis=1).loc[idx], w.loc[idx])
    m2t, _ = fit_wls(df.loc[idx, "_tev_leven"], pd.concat([D, Cs[basis], huidig], axis=1).loc[idx], w.loc[idx])

    for i, cat in enumerate(cats[1:]):
        k = f"cat_{i}"
        b1p, b2p, b1t, b2t = m1p.params[k], m2p.params[k], m1t.params[k], m2t.params[k]
        rows.append({"Vroeger": naam, "Categorie (t.o.v. referentie)": f"{cat} (t.o.v. {ref})",
                     "OR pessimisme - model 1": np.exp(b1p), "p model 1": m1p.pvalues[k],
                     "OR pessimisme - model 2": np.exp(b2p), "p model 2": m2p.pvalues[k],
                     "% effect pessimisme via huidige situatie": 100 * (1 - b2p / b1p) if m1p.pvalues[k] < 0.05 else np.nan,
                     "Verschil tevredenheid (punten) - model 1": b1t,
                     "Verschil tevredenheid (punten) - model 2": b2t,
                     "% effect tevredenheid via huidige situatie": 100 * (1 - b2t / b1t) if m1t.pvalues[k] < 0.05 else np.nan,
                     "n": n})

bewaar("8b_Doorwerking_jeugd", "Werkt de jeugd door in tevredenheid en toekomstblik, en loopt dat via de huidige situatie?",
       "Extra: kansengelijkheid - van jeugd naar nu", pd.DataFrame(rows),
       "Gewogen regressies. Model 1: jeugdkenmerk + leeftijd, geslacht. Model 2: + opleiding, inkomen, aantal financiële "
       "onzekerheden, aantal domeinen onvoldoende. '% effect via huidige situatie' = hoeveel kleiner het verband wordt in "
       "model 2 (alleen berekend als model 1 significant is). Indicatief, geen causale mediatie-analyse.",
       fmt={"OR pessimisme - model 1": "0.00", "OR pessimisme - model 2": "0.00", "p model 1": "0.000", "p model 2": "0.000",
            "% effect pessimisme via huidige situatie": "0", "% effect tevredenheid via huidige situatie": "0",
            "Verschil tevredenheid (punten) - model 1": "0.00", "Verschil tevredenheid (punten) - model 2": "0.00"},
       visual="Twee staven per jeugdkenmerk (model 1 vs 2)")

## Export

In [ ]:
# ── Excel voor de vormgever ─────────────────────────────────────────────────
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

inhoud = pd.DataFrame([{"Tabblad": k, "Titel": v["titel"], "Subvraag": v["vraag"], "Suggestie visual": v["visual"]}
                       for k, v in tabellen.items()])
definities = [
    ("Bron", "Werkbestand BW en kansengelijkheid (Reactiestatus ≠ 'Niet geantwoord')"),
    ("Aantal respondenten", len(df)),
    ("Weging", "Alle % en gemiddelden gewogen met 'weegfactor' (provincie x leeftijd x opleiding). n = ongewogen."),
    ("Tevreden", f"Rapportcijfer {TEVREDEN_GRENS} of hoger voor 'Uw leven in het algemeen'"),
    ("Onvoldoende / niet op orde", f"Rapportcijfer lager dan {ONVOLDOENDE_GRENS} voor een domein"),
    ("Pessimistisch", "Verwacht dat het over 5 jaar (een beetje of veel) slechter gaat"),
    ("Hoopvol/stabiel", "Verwacht dat het over 5 jaar hetzelfde of beter gaat"),
    ("Veel invloed (grip)", "Antwoord 'Veel invloed' op 'Hoeveel invloed denkt u dat u heeft op ...'"),
    ("Let op", "Percentages op basis van minder dan ca. 50 respondenten (kolom n) voorzichtig interpreteren."),
]
kop = PatternFill("solid", fgColor="DDE7F0")

with pd.ExcelWriter(EXCEL_OUT, engine="openpyxl") as xw:
    inhoud.to_excel(xw, sheet_name="0_Inhoud", startrow=4, index=False)
    ws = xw.sheets["0_Inhoud"]
    ws["A1"] = "Spoor 1 - Tevredenheid met het leven, toekomstperspectief en grip"
    ws["A1"].font = Font(bold=True, size=14)
    ws["A2"] = "Cijfers voor het informatieproduct. Elk tabblad = één tabel/figuur."
    r0 = 5 + len(inhoud) + 2
    ws.cell(r0, 1, "Definities").font = Font(bold=True, size=12)
    for i, (k, v) in enumerate(definities, start=1):
        ws.cell(r0 + i, 1, k).font = Font(bold=True)
        ws.cell(r0 + i, 2, v)
    for j, breedte in enumerate([32, 80, 70, 40], start=1):
        ws.cell(5, j).font, ws.cell(5, j).fill = Font(bold=True), kop
        ws.column_dimensions[get_column_letter(j)].width = breedte

    for naam, t in tabellen.items():
        d = t["df"]
        d.to_excel(xw, sheet_name=naam, startrow=4, index=False)
        ws = xw.sheets[naam]
        ws["A1"], ws["A2"], ws["A3"] = t["titel"], t["vraag"], t["toelichting"]
        ws["A4"] = "Bron: Werkbestand BW en kansengelijkheid, gewogen. n = ongewogen aantal."
        ws["A1"].font, ws["A2"].font, ws["A4"].font = Font(bold=True, size=14), Font(italic=True), Font(size=9, color="666666")
        for j, col in enumerate(d.columns, start=1):
            h = ws.cell(5, j)
            h.font, h.fill, h.alignment = Font(bold=True), kop, Alignment(wrap_text=True, vertical="top")
            fmt = t["fmt"].get(col)
            if fmt is None and (col == "n" or str(col).startswith("n ")):
                fmt = "0"
            elif fmt is None and pd.api.types.is_float_dtype(d[col]):
                fmt = "0.0"
            if fmt:
                for i in range(len(d)):
                    ws.cell(6 + i, j).number_format = fmt
            breedte = max([len(str(col)) // 2 + 2] + [len(str(v)) for v in d[col].head(60)] + [8])
            ws.column_dimensions[get_column_letter(j)].width = min(max(breedte + 2, 10), 60)
        ws.row_dimensions[5].height = 45
        ws.freeze_panes = ws.cell(6, 2)
        if len(d) and str(d.iloc[0, 0]).startswith("n (ongewogen)"):
            for j in range(1, d.shape[1] + 1):
                ws.cell(6, j).number_format = "0"
                ws.cell(6, j).font = Font(italic=True, color="666666")

print(f"Excel geschreven: {EXCEL_OUT} ({len(tabellen) + 1} tabbladen)")

In [ ]:
# ── Open antwoorden voor citaten (INTERN, niet naar GitHub/vormgever zonder anonimiteitscheck) ─
open_col = first_col(df, "Zijn er andere onderwerpen waarover u zorgen voor de toekomst heeft")
if open_col:
    s = df[open_col].astype("string").str.strip()
    m = s.notna() & (s.str.len() > 2) & ~s.str.lower().isin(["nee", "geen", "nvt", "n.v.t.", "-", "nee.", "geen."])
    open_df = pd.DataFrame({"Groep": df.loc[m, "_groep"], "Leeftijdsgroep": df.loc[m, "Leeftijdsgroep"],
                            "Tevredenheid leven": df.loc[m, "_tev_leven"],
                            "Verwachting leven (1-5)": df.loc[m, "_toe_leven"], "Antwoord": s[m]})
    open_df["_volgorde"] = open_df["Groep"].map({g: i for i, g in enumerate(GROEPEN[::-1])}).fillna(9)
    open_df.sort_values(["_volgorde", "Leeftijdsgroep"]).drop(columns="_volgorde").to_excel(OPEN_OUT, index=False)
    print(f"Open antwoorden: {m.sum()} -> {OPEN_OUT}")